# DyslexAI — Notebook 04b: Baseline CNN, two preprocessing pipelines

**Jira: DYS-18, DYS-27, DYS-28**

---

## What we are testing

We have two ways of preparing the images, and we do not know which is better. This notebook
runs the same model on both and compares them fairly.

**Pipeline A — ink crop.** Find the pencil marks, put a square box around them, use the real
paper to make it square. The letter fills the picture. Nothing gets stretched.

**Pipeline B — padding.** Use the whole photo, add blank space to make it square, then shrink
it. The letter stays a small mark in a big empty square.

## The number we care about

Not the accuracy on its own. Accuracy can be faked.

We already know you can guess the class reasonably well from just the *size and shape numbers*
of each image, without looking at the handwriting at all. That score is the **floor**.

So the number that matters is:

```
what the model scored  minus  the floor  =  what it actually learned from handwriting
```

A model scoring 80% where the floor is 62% learned 18 points. A model scoring 63% where the
floor is 56% learned 7 points. The first one is better, even though both "beat the floor".

## How we keep it fair

The same images go into train, validation and test for both pipelines. Same model, same
settings, same random seed. **Only the picture preparation changes.** So any difference has to
come from that.

---

## Before you run this

You need in `MyDrive/DyslexAI/archives/`:

- `Complete_And_Balance_Hand-written_Dataset.rar` (the original photos)
- `Autocrop_No.zip` and `Autocrop_Yes.zip` (made by notebook 01c)

And in `data/interim/`: `dataset_a_master_metadata.csv` and `dataset_a_autocrop_metadata.csv`.

**Turn the GPU on:** Runtime → Change runtime type → T4 GPU. On GPU this takes about 10
minutes. On CPU, over two hours.

---
# STAGE 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, random, subprocess, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | device: {DEVICE}')
if not torch.cuda.is_available():
    print('WARNING: no GPU. This will take hours instead of minutes.')

DRIVE       = Path('/content/drive/MyDrive/DyslexAI')
ARCHIVE_DIR = DRIVE / 'archives'
INTERIM_DIR = DRIVE / 'data' / 'interim'
REPORTS_DIR = DRIVE / 'reports'
FIGURES_DIR = DRIVE / 'results' / 'figures'
EXP_DIR     = DRIVE / 'experiments'
RAW_DIR     = Path('/content/raw')

for d in [REPORTS_DIR, FIGURES_DIR, EXP_DIR, RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); os.environ['PYTHONHASHSEED'] = str(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
print('ready, seed =', SEED)

In [ ]:
# ---- Unpack both versions of the images ------------------------------------
!apt-get -qq install -y libarchive-tools > /dev/null 2>&1

def unpack(archive, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    if not any(dest.rglob('*.jpeg')) and not any(dest.rglob('*.jpg')):
        subprocess.run(['bsdtar', '-xf', str(ARCHIVE_DIR / archive), '-C', str(dest)],
                       capture_output=True, check=False)
    return dest

ORIG_ROOT = unpack('Complete_And_Balance_Hand-written_Dataset.rar', RAW_DIR / 'original')
CROP_ROOT = RAW_DIR / 'autocrop'
unpack('Autocrop_No.zip',  CROP_ROOT)
unpack('Autocrop_Yes.zip', CROP_ROOT)

def find_dir(root, cls, minimum=100):
    best = None
    for d in [root] + [p for p in Path(root).rglob('*') if p.is_dir()]:
        n = sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT)
        if n >= minimum and cls.lower() in d.name.lower() and (best is None or n > best[0]):
            best = (n, d)
    if best is None:
        raise FileNotFoundError(f'no folder for {cls} under {root}')
    return best[1]

ORIG_DIRS = {c: find_dir(ORIG_ROOT, c) for c in ['No', 'Yes']}
CROP_DIRS = {c: find_dir(CROP_ROOT, c) for c in ['No', 'Yes']}
print('originals:', {c: str(d) for c, d in ORIG_DIRS.items()})
print('autocrops:', {c: str(d) for c, d in CROP_DIRS.items()})

### Build one split, used by both pipelines

We clean the data the same way as always — one file per original image, contradictory labels
removed — then split it once. Both pipelines then use that exact same split.

In [ ]:
auto = pd.read_csv(INTERIM_DIR / 'dataset_a_autocrop_metadata.csv')

# Clean: drop files whose original appeared under both labels, then keep one
# file per original image (the duplicates are the same photo copied).
clean = (auto[auto['is_cross_class_conflict'] == 0]
         .sort_values(['duplicate_group', 'original_filename'])
         .drop_duplicates('duplicate_group')
         .reset_index(drop=True))

# Paths for BOTH pipelines, same rows
clean['crop_path'] = clean.apply(
    lambda r: str(CROP_DIRS[r['class_label']] / r['original_filename']), axis=1)
clean['orig_path'] = clean.apply(
    lambda r: str(ORIG_DIRS[r['class_label']] / r['original_filename']), axis=1)

for col in ['crop_path', 'orig_path']:
    missing = [p for p in clean[col] if not Path(p).exists()]
    assert not missing, f'{len(missing)} missing in {col}, e.g. {missing[:2]}'

print(f'all images      : {len(auto)}')
print(f'after cleaning  : {len(clean)}')
print(clean['class_label'].value_counts().sort_index().to_string())

In [ ]:
from sklearn.model_selection import train_test_split

tr, tmp = train_test_split(clean, test_size=0.30,
                           stratify=clean['class_label'], random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50,
                           stratify=tmp['class_label'], random_state=SEED)

clean['split'] = 'train'
clean.loc[va.index, 'split'] = 'val'
clean.loc[te.index, 'split'] = 'test'

print(pd.crosstab(clean['split'], clean['class_label'], margins=True).to_string())

test_counts = clean.query('split == "test"')['class_label'].value_counts()
MAJORITY = test_counts.max() / test_counts.sum()
print(f'\nAlways guessing the commonest class on the test set = {MAJORITY:.1%} accuracy')
print('(balanced accuracy for that strategy is 50.0% — that is our chance level)')

---
# STAGE 1 — The Floor

Before training anything, work out what score you can reach **without looking at the pictures**.

We give a simple program only the numbers describing each image — how big it is, how big the
pencil marks are, how much padding was needed — and ask it to guess the class.

Whatever it scores is the floor. Any model that scores near it has not proved it learned
anything from handwriting.

We work this out separately for each pipeline, because they expose different numbers.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

g = clean.copy()
g['orig_aspect']     = g['orig_width'] / g['orig_height']
g['orig_distortion'] = g['orig_aspect'].apply(lambda a: max(a, 1/a))
g['orig_area']       = g['orig_width'] * g['orig_height']
g['orig_pad_frac']   = 1 - (g['orig_width'] * g['orig_height']) / \
                       (g[['orig_width', 'orig_height']].max(axis=1) ** 2)
g['ink_distortion']  = g['ink_aspect'].apply(lambda a: max(a, 1/a))

FEATURES = {
    'A: ink crop': ['crop_side', 'ink_width', 'ink_height', 'ink_aspect',
                    'ink_fill', 'pad_fraction', 'ink_distortion'],
    'B: padding' : ['orig_width', 'orig_height', 'orig_aspect',
                    'orig_distortion', 'orig_area', 'orig_pad_frac'],
}

def geometry_floor(feats):
    trn = g[g['split'] != 'test']; tst = g[g['split'] == 'test']
    ytr = (trn['class_label'] == 'Yes').astype(int)
    yte = (tst['class_label'] == 'Yes').astype(int)
    sc  = StandardScaler().fit(trn[feats])
    best = 0.0
    for clf in [LogisticRegression(max_iter=2000, class_weight='balanced'),
                RandomForestClassifier(n_estimators=300, random_state=SEED,
                                       class_weight='balanced')]:
        clf.fit(sc.transform(trn[feats]), ytr)
        best = max(best, balanced_accuracy_score(yte, clf.predict(sc.transform(tst[feats]))))
    return round(best, 3)

FLOOR = {name: geometry_floor(f) for name, f in FEATURES.items()}
for k, v in FLOOR.items():
    print(f'{k:<14} floor = {v:.3f}   (chance = 0.500)')

---
# STAGE 2 — The Two Pipelines

**Pipeline A** takes the already-square ink crops and just shrinks them to 224×224. No
stretching, because they are already square.

**Pipeline B** takes the original photo, adds blank paper-coloured border to make it square,
then shrinks it. Nothing is stretched either, but most of the picture is empty page.

In [ ]:
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

class PadToSquare:
    """Add border to make the picture square, using the paper's own colour."""
    def __call__(self, img):
        w, h = img.size
        if w == h:
            return img
        arr = np.asarray(img); c = 8
        if arr.ndim == 3 and min(arr.shape[:2]) >= 2 * c:
            corners = np.concatenate([arr[:c, :c].reshape(-1, 3), arr[:c, -c:].reshape(-1, 3),
                                      arr[-c:, :c].reshape(-1, 3), arr[-c:, -c:].reshape(-1, 3)])
            fill = tuple(int(v) for v in np.median(corners, axis=0))
        else:
            fill = (166, 166, 166)
        side = max(w, h)
        canvas = Image.new('RGB', (side, side), fill)
        canvas.paste(img, ((side - w) // 2, (side - h) // 2))
        return canvas

BASE = [T.ToTensor(), T.Normalize(MEAN, STD)]

TRANSFORMS = {
    'A: ink crop': T.Compose([T.Lambda(lambda im: im.convert('RGB')),
                              T.Resize((IMG_SIZE, IMG_SIZE))] + BASE),
    'B: padding' : T.Compose([T.Lambda(lambda im: im.convert('RGB')),
                              PadToSquare(),
                              T.Resize((IMG_SIZE, IMG_SIZE))] + BASE),
}
PATH_COL = {'A: ink crop': 'crop_path', 'B: padding': 'orig_path'}

for k, v in TRANSFORMS.items():
    print(f'{k}: {len(v.transforms)} steps, reads {PATH_COL[k]}')

In [ ]:
# ---- Look at one image through both pipelines ------------------------------
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

def show(t):
    m = torch.tensor(MEAN).view(3,1,1); s = torch.tensor(STD).view(3,1,1)
    return (t.cpu() * s + m).clamp(0,1).permute(1,2,0)

sample = clean.sample(5, random_state=SEED)
fig, ax = plt.subplots(2, 5, figsize=(13, 5.5))
for col, (_, r) in enumerate(sample.iterrows()):
    ax[0, col].imshow(show(TRANSFORMS['A: ink crop'](Image.open(r['crop_path']))))
    ax[0, col].set_title('A: ink crop', fontsize=8); ax[0, col].axis('off')
    ax[1, col].imshow(show(TRANSFORMS['B: padding'](Image.open(r['orig_path']))))
    ax[1, col].set_title('B: padding', fontsize=8); ax[1, col].axis('off')
fig.suptitle('Same images, two ways of preparing them', fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'A_pipeline_comparison.png', bbox_inches='tight')
plt.show()
print('Notice how much of the bottom row is blank paper.')

---
# STAGE 3 — The Model and Training

Small network, sized to the data: three convolution blocks, about 24,000 settings for 432
training images. A bigger one would just memorise them.

We pick the best version of the model using **balanced accuracy** on the validation set, not
plain accuracy. Plain accuracy is misleading here: guessing "Yes" every time already scores
about 66%.

In [ ]:
CLASS_TO_IDX = {'No': 0, 'Yes': 1}

class HandwritingDataset(Dataset):
    def __init__(self, df, split, transform, path_col):
        self.df = df[df['split'] == split].reset_index(drop=True)
        self.transform, self.path_col = transform, path_col
        self.labels = self.df['class_label'].map(CLASS_TO_IDX).values
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = Image.open(self.df.iloc[i][self.path_col])
        return self.transform(img), int(self.labels[i])


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        def block(i, o):
            return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o),
                    nn.ReLU(inplace=True), nn.MaxPool2d(2)]
        self.features = nn.Sequential(*block(3,16), *block(16,32), *block(32,64))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout),
                                        nn.Linear(64, num_classes))
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


def evaluate(model, loader, criterion):
    model.eval(); tot, n, P, Y = 0.0, 0, [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x); tot += criterion(out, y).item() * len(y); n += len(y)
            P.append(out.argmax(1).cpu()); Y.append(y.cpu())
    p, t = torch.cat(P).numpy(), torch.cat(Y).numpy()
    rec = [ (p[t==c]==c).mean() if (t==c).any() else 0.0 for c in (0,1) ]
    return {'loss': tot/n, 'accuracy': float((p==t).mean()),
            'balanced_accuracy': float(np.mean(rec)),
            'recall_No': float(rec[0]), 'recall_Yes': float(rec[1]),
            'preds': p, 'truth': t}


def run_experiment(name, transform, path_col, epochs=40, lr=1e-3, batch=32):
    set_seed(SEED)
    g_ = torch.Generator(); g_.manual_seed(SEED)
    loaders = {}
    for sp in ['train', 'val', 'test']:
        ds = HandwritingDataset(clean, sp, transform, path_col)
        loaders[sp] = DataLoader(ds, batch_size=batch, shuffle=(sp=='train'),
                                 num_workers=2, pin_memory=torch.cuda.is_available(),
                                 generator=g_ if sp=='train' else None)

    model = SimpleCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    hist, best, best_ep, best_state = [], -1.0, -1, None
    t0 = time.time()
    for ep in range(1, epochs+1):
        model.train(); tot, corr, n = 0.0, 0, 0
        for x, y in loaders['train']:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); out = model(x); loss = criterion(out, y)
            loss.backward(); opt.step()
            tot += loss.item()*len(y); corr += (out.argmax(1)==y).sum().item(); n += len(y)
        v = evaluate(model, loaders['val'], criterion)
        hist.append({'epoch':ep,'train_loss':tot/n,'train_accuracy':corr/n,
                     'val_loss':v['loss'],'val_accuracy':v['accuracy'],
                     'val_balanced_accuracy':v['balanced_accuracy']})
        if v['balanced_accuracy'] > best:
            best, best_ep = v['balanced_accuracy'], ep
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        if ep == 1 or ep % 10 == 0 or ep == epochs:
            print(f'  epoch {ep:>3}/{epochs}  train_loss={tot/n:.4f}  '
                  f'val_bal_acc={v["balanced_accuracy"]:.3f}')

    model.load_state_dict(best_state)
    test = evaluate(model, loaders['test'], criterion)
    print(f'  finished in {time.time()-t0:.0f}s | best epoch {best_ep} | '
          f'TEST balanced accuracy {test["balanced_accuracy"]:.3f}')
    return {'name':name,'history':pd.DataFrame(hist),'test':test,
            'best_epoch':best_ep,'seconds':round(time.time()-t0)}

---
# STAGE 4 — Run Both

About five minutes each on a GPU.

In [ ]:
results = {}
for name in ['A: ink crop', 'B: padding']:
    print(f'\n=== {name}  (floor = {FLOOR[name]:.3f}) ===')
    results[name] = run_experiment(name, TRANSFORMS[name], PATH_COL[name])

---
# STAGE 5 — Results

The important column is the last one: **how far above the floor**. That is the part of the
score that had to come from the handwriting itself.

In [ ]:
rows = []
for name, r in results.items():
    t = r['test']
    rows.append({
        'pipeline'   : name,
        'floor'      : FLOOR[name],
        'accuracy'   : round(t['accuracy'], 3),
        'balanced'   : round(t['balanced_accuracy'], 3),
        'recall_No'  : round(t['recall_No'], 3),
        'recall_Yes' : round(t['recall_Yes'], 3),
        'above_floor': round(t['balanced_accuracy'] - FLOOR[name], 3),
        'best_epoch' : r['best_epoch'],
        'seconds'    : r['seconds'],
    })
table = pd.DataFrame(rows).sort_values('above_floor', ascending=False)
print(table.to_string(index=False))

table.to_csv(REPORTS_DIR / 'pipeline_comparison_results.csv', index=False)
print(f'\n[saved] {REPORTS_DIR / "pipeline_comparison_results.csv"}')

print('\n--- what this means ---')
for _, r in table.iterrows():
    if r['above_floor'] <= 0.02:
        verdict = 'learned nothing we can prove — score is reachable without the pictures'
    elif r['above_floor'] < 0.08:
        verdict = 'learned a little, but not much more than the numbers alone give'
    else:
        verdict = 'learned something real from the handwriting'
    print(f"  {r['pipeline']:<14} {r['balanced']:.3f} vs floor {r['floor']:.3f}  ->  {verdict}")

best = table.iloc[0]
print(f"\nBest pipeline: {best['pipeline']}  ({best['above_floor']:+.3f} above its floor)")

In [ ]:
# ---- Training curves and comparison figure ---------------------------------
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))

for name, r in results.items():
    h = r['history']
    ax[0].plot(h['epoch'], h['train_loss'], label=f'{name} train')
    ax[0].plot(h['epoch'], h['val_loss'], '--', label=f'{name} val')
    ax[1].plot(h['epoch'], h['val_balanced_accuracy'], label=name)

ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].set_title('Loss')
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('balanced accuracy')
ax[1].set_title('Validation balanced accuracy'); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)

x = np.arange(len(table))
ax[2].bar(x - 0.2, table['floor'], width=0.4, label='floor (no pictures)', color='#aaa')
ax[2].bar(x + 0.2, table['balanced'], width=0.4, label='CNN', color='#47a')
ax[2].axhline(0.5, color='k', ls=':', lw=1)
ax[2].set_xticks(x); ax[2].set_xticklabels(table['pipeline'], fontsize=8)
ax[2].set_ylabel('balanced accuracy'); ax[2].set_ylim(0, 1)
ax[2].set_title('Model vs its floor'); ax[2].legend(fontsize=8)
for i, (f, b) in enumerate(zip(table['floor'], table['balanced'])):
    ax[2].text(i-0.2, f, f'{f:.2f}', ha='center', va='bottom', fontsize=8)
    ax[2].text(i+0.2, b, f'{b:.2f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Ink crop vs padding — same model, same split, same seed', fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'A_pipeline_cnn_comparison.png', bbox_inches='tight')
plt.show()
print(f'[saved] {FIGURES_DIR / "A_pipeline_cnn_comparison.png"}')

---
# STAGE 6 — Report

In [ ]:
report = f"""
{'='*70}
INK CROP vs PADDING - BASELINE CNN COMPARISON
{'='*70}
Generated : {datetime.now().isoformat(timespec='seconds')}
Seed      : {SEED}
Jira      : DYS-18, DYS-27, DYS-28

WHAT WE DID
  Ran the same small CNN twice. Same images in train/val/test, same
  settings, same random seed. The only difference was how the pictures
  were prepared.

  A: ink crop  - square box around the pencil marks, nothing stretched
  B: padding   - whole photo, blank border added to make it square

DATA
  images after cleaning : {len(clean)}
  train / val / test    : {(clean['split']=='train').sum()} / {(clean['split']=='val').sum()} / {(clean['split']=='test').sum()}
  test set class split  : {clean.query('split=="test"')['class_label'].value_counts().to_dict()}

THE FLOOR
  How well you can guess the class from the size and shape numbers
  alone, without looking at any picture. Chance is 0.500.

  A: ink crop  {FLOOR['A: ink crop']:.3f}
  B: padding   {FLOOR['B: padding']:.3f}

RESULTS
{table.to_string(index=False)}

  'above_floor' is the part of the score that had to come from the
  handwriting. It is the number to quote, not the raw accuracy.

LIMITATIONS
  * No participant IDs, so we cannot separate children between training
    and testing. No result here is writer-independent.
  * Small test set, so differences of a few points are not reliable.
  * Single run each. Repeat with several seeds before drawing a firm
    conclusion.
  * Yes/No are the labels the dataset came with. They are not diagnoses.
{'='*70}
""".strip()

print(report)
(REPORTS_DIR / 'pipeline_comparison_report.txt').write_text(report)
print(f'\n[saved] {REPORTS_DIR / "pipeline_comparison_report.txt"}')

---
# What to tell the supervisor

## What we did

Ran the same model twice, changing only how the pictures were prepared. Everything else was
held identical so the comparison is fair.

## Why we report two numbers, not one

An accuracy figure on its own can be misleading. We measured how well the class can be guessed
from image size and shape alone — no handwriting involved. That is the floor. A model that
scores close to the floor has not shown it learned anything from the writing.

## What to say

> We compared two ways of preparing the images using an identical model and split. For each we
> also measured how much of the score is reachable without looking at the handwriting at all,
> and we report the difference. This tells us how much the model actually learned from the
> writing rather than from the shape of the picture.

## Next steps

Once we know which pipeline is better, the same comparison runs for ResNet18, ResNet50, VGG16
and EfficientNet. Every result gets reported against its floor.

Before trusting any of it, repeat the winning pipeline with 3–5 different random seeds. With a
test set this small, one run can move several points by luck.